# Train YOLOv11 on the merged fire/smoke dataset

Primary training entry point. Run on the **4090 laptop** after preparing the merged dataset.

**Pre-flight checklist** (the difference between training in 60 minutes and not training at all):
- [ ] CUDA torch installed: `pip install -r requirements-cuda.txt --index-url https://download.pytorch.org/whl/cu121`
- [ ] Plugged into wall power, set Windows power profile to "Best Performance"
- [ ] `nvidia-smi` shows the RTX 4090 and >=14 GB free
- [ ] `data/merged/data.yaml` exists (run `scripts/prepare_dataset.py` first)
- [ ] At least 80 GB free disk

**Colab portability:** to run this notebook on Colab instead, mount Drive, replace `ROOT` with the Drive path containing the merged dataset, install ultralytics, and skip the local-environment cell.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('project root:', ROOT)

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('CUDA not available — install requirements-cuda.txt before training')

In [ ]:
DATA_YAML = ROOT / 'data' / 'merged' / 'data.yaml'
assert DATA_YAML.exists(), f'no merged dataset at {DATA_YAML} — run scripts/prepare_dataset.py first'
print(DATA_YAML.read_text())

## Stage 1 — baseline training

`yolo11s.pt` start, 80 epochs, AdamW + cosine LR + AMP. Cache to RAM if you have >=32 GB; switch to `disk` otherwise.

In [ ]:
from ultralytics import YOLO
RUN_NAME = 'v11s-baseline'
model = YOLO('yolo11s.pt')
results = model.train(
    data=str(DATA_YAML),
    epochs=80,
    imgsz=640,
    batch=32,
    device=0,
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    amp=True,
    cache='ram',
    project=str(ROOT / 'runs' / 'firesmoke'),
    name=RUN_NAME,
    plots=True,
)
print('best weights:', model.trainer.best)

## Stage 2 — validate on the held-out test split

The test split was kept out of training and val. This is the only honest number for "will it work in production".

In [ ]:
best = ROOT / 'runs' / 'firesmoke' / RUN_NAME / 'weights' / 'best.pt'
from ultralytics import YOLO
test_model = YOLO(str(best))
test_metrics = test_model.val(data=str(DATA_YAML), split='test', imgsz=640, device=0)
print(test_metrics.box.map50, test_metrics.box.map)

## Stage 3 — copy `best.pt` into `models/`

This is what the demo's inference service loads.

In [ ]:
import shutil
MODELS = ROOT / 'models'
MODELS.mkdir(exist_ok=True)
shutil.copy2(best, MODELS / 'best.pt')
print('->', MODELS / 'best.pt')

## Stage 4 (optional) — negative-class fine-tune

After the baseline is in place, collect industrial false-positive footage (welding, steam, dust, sunset, orange clothing) into `data/raw/industrial_negatives/` with **empty `.txt` label files**. Re-run `prepare_dataset.py` to merge them in, then continue training from the baseline `best.pt` at a 10× lower learning rate.

In [ ]:
# Uncomment after preparing industrial negatives:
# from ultralytics import YOLO
# m2 = YOLO(str(best))
# m2.train(
#     data=str(DATA_YAML), epochs=20, imgsz=640, batch=32,
#     lr0=0.0001, optimizer='AdamW', cos_lr=True, amp=True,
#     project=str(ROOT / 'runs' / 'firesmoke'), name='v11s-neg-tuned')

## Stage 5 — export for deployment

In [ ]:
from ultralytics import YOLO
m = YOLO(str(MODELS / 'best.pt'))
m.export(format='onnx', opset=12, simplify=True)
# m.export(format='engine', half=True, device=0)  # NVIDIA edge (Jetson, etc.)
# m.export(format='openvino')                     # Intel CPU/GPU